# Day 061 — Exercise 2: JsonFormatter

Plain-text log lines like `2026-01-01 12:00:00 | app | INFO | Request received` are human-readable but hard to query in production. Tools like Datadog, Loki, and CloudWatch work much better with **structured logs** — one JSON object per line, where each field can be indexed and filtered.

A custom `logging.Formatter` subclass controls what each log record looks like. Override `format(record) -> str` to return any string you want.

In [ ]:
import json
import logging
from datetime import datetime


## Task

Implement `JsonFormatter(logging.Formatter)` with a `format` method that returns:

```json
{"timestamp": "2026-01-01T12:00:00", "level": "INFO",
 "logger": "myapp", "message": "Hello world"}
```

Field sources:
- `timestamp`: `datetime.fromtimestamp(record.created).isoformat()`
- `level`: `record.levelname`
- `logger`: `record.name`
- `message`: `record.getMessage()` (handles `%s` formatting args)
- `exc` (only if `record.exc_info`): `self.formatException(record.exc_info)`

## Your Implementation

In [ ]:
class JsonFormatter(logging.Formatter):
    """Emit one JSON object per log record.

    Each call to format(record) returns a string like:
    {"timestamp": "2026-01-01T12:00:00", "level": "INFO",
     "logger": "myapp", "message": "Hello world"}

    If the record has exc_info (an exception), also include:
    {"exc": "<formatted traceback string>"}

    Use datetime.fromtimestamp(record.created).isoformat() for the timestamp.
    Use record.getMessage() for the message (handles % formatting).
    Use self.formatException(record.exc_info) for the traceback string.
    """

    def format(self, record: logging.LogRecord) -> str:
        # TODO: build entry dict, add exc key if record.exc_info, return json.dumps(entry)
        raise NotImplementedError


In [ ]:
class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        entry = {
            "timestamp": datetime.fromtimestamp(record.created).isoformat(),
            "level":     record.levelname,
            "logger":    record.name,
            "message":   record.getMessage(),
        }
        if record.exc_info:
            entry["exc"] = self.formatException(record.exc_info)
        return json.dumps(entry)


## Automated checks

In [ ]:
score, total = 0, 5
try:
    formatter = JsonFormatter()

    # helper: make a real LogRecord
    def make_record(msg, level=logging.INFO, name="test"):
        r = logging.LogRecord(
            name=name, level=level, pathname="", lineno=0,
            msg=msg, args=(), exc_info=None,
        )
        return r

    # output is valid JSON
    rec1 = make_record("hello world")
    output = formatter.format(rec1)
    data = json.loads(output)
    score += 1; print("\u2705 format() returns valid JSON")

    # required fields present
    for field in ("timestamp", "level", "logger", "message"):
        assert field in data, f"Missing field: {field}"
    score += 1; print("\u2705 output has timestamp, level, logger, message")

    # level name is correct
    assert data["level"] == "INFO", f"Expected 'INFO', got {data['level']!r}"
    score += 1; print("\u2705 level field contains levelname string")

    # message is correct
    assert data["message"] == "hello world", f"Got {data['message']!r}"
    score += 1; print("\u2705 message field contains the log message")

    # exception adds 'exc' key
    try:
        raise ValueError("oops")
    except ValueError:
        import sys
        exc_info = sys.exc_info()
    rec2 = logging.LogRecord(
        name="test", level=logging.ERROR, pathname="", lineno=0,
        msg="error occurred", args=(), exc_info=exc_info,
    )
    data2 = json.loads(formatter.format(rec2))
    assert "exc" in data2, f"Expected 'exc' key for exception record"
    assert "ValueError" in data2["exc"]
    score += 1; print("\u2705 exception record includes 'exc' traceback field")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
class JsonFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        entry = {
            "timestamp": datetime.fromtimestamp(record.created).isoformat(),
            "level":     record.levelname,
            "logger":    record.name,
            "message":   record.getMessage(),
        }
        if record.exc_info:
            entry["exc"] = self.formatException(record.exc_info)
        return json.dumps(entry)
```

**Why `record.getMessage()` not `record.msg`?** `record.msg` is the raw format string (e.g. `'user %s logged in'`). `getMessage()` applies the args: `'user alice logged in'`. Always use `getMessage()` in formatters.

**Why `self.formatException(record.exc_info)` not `traceback.format_exc()`?** `formatException` uses the exc_info stored in the record — safe across threads and callbacks. `traceback.format_exc()` reads the current thread's exception state, which may differ.

</details>